# **Tahap 2 - Cleaning**

## Import Library

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

## Path Konfigurasi

In [2]:
BASE_DIR      = Path(__file__).resolve().parent.parent \
                if "__file__" in dir() else Path.cwd().parent
 
EXTRACTED_DIR = BASE_DIR / "data" / "extracted"
CLEANED_DIR   = BASE_DIR / "data" / "cleaned"
 
if not EXTRACTED_DIR.exists():
    raise FileNotFoundError(
        f"Folder data/extracted tidak ditemukan: {EXTRACTED_DIR}\n"
        f"Jalankan dulu: python src/01_extract_pdf.py"
    )
 
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

## Constanta

In [3]:
PROVINSI_RESMI = {
    "Aceh", "Sumatera Utara", "Sumatera Barat", "Riau", "Jambi",
    "Sumatera Selatan", "Bengkulu", "Lampung", "Kepulauan Bangka Belitung",
    "Kepulauan Riau", "DKI Jakarta", "Jawa Barat", "Jawa Tengah",
    "DI Yogyakarta", "Jawa Timur", "Banten", "Bali",
    "Nusa Tenggara Barat", "Nusa Tenggara Timur", "Kalimantan Barat",
    "Kalimantan Tengah", "Kalimantan Selatan", "Kalimantan Timur",
    "Kalimantan Utara", "Sulawesi Utara", "Sulawesi Tengah",
    "Sulawesi Selatan", "Sulawesi Tenggara", "Gorontalo", "Sulawesi Barat",
    "Maluku", "Maluku Utara", "Papua Barat", "Papua",
}

PROV_NORMALIZE = {
    "Dki Jakarta"              : "DKI Jakarta",
    "D.K.I. Jakarta"           : "DKI Jakarta",
    "Jakarta"                  : "DKI Jakarta",
    "Di Yogyakarta"            : "DI Yogyakarta",
    "D.I. Yogyakarta"          : "DI Yogyakarta",
    "DI. Yogyakarta"           : "DI Yogyakarta",
    "Yogyakarta"               : "DI Yogyakarta",
    "Kep. Bangka Belitung"     : "Kepulauan Bangka Belitung",
    "Bangka Belitung Islands"  : "Kepulauan Bangka Belitung",
    "Riau Islands"             : "Kepulauan Riau",
    "Papua Barat Daya"         : "Papua Barat",
    "Papua Selatan"            : "Papua",
    "Papua Tengah"             : "Papua",
    "Papua Pegunungan"         : "Papua",
}

POPULATION_GROWTH_RATE = 0.0117  

OUTLIER_MAX = 200_000

## Helper Function 

In [4]:
def normalize_province_name(name: str) -> str:
    if not isinstance(name, str):
        return name
 
    name = name.strip()
 
    if name.isupper():
        name = name.title()

    name = re.sub(r'\bDki\b', 'DKI', name)
    name = re.sub(r'\bDi\b',  'DI',  name)
    name = re.sub(r'\bKep\b', 'Kep', name)
    name = re.sub(r'\bNtb\b', 'NTB', name)
    name = re.sub(r'\bNtt\b', 'NTT', name)
 
    return PROV_NORMALIZE.get(name, name)
 
def print_section(title: str) -> None:
    print(f"\n{'=' * 60}")
    print(title)
    print('=' * 60)
 
def print_validation(label: str, ok: bool, detail: str = "") -> None:
    icon = "✅" if ok else "⚠️ "
    msg  = f"  {icon} {label}"
    if detail:
        msg += f" — {detail}"
    print(msg)
 

## Bagian 1 - Cleaning Crime Data

In [ ]:
def clean_crime(df: pd.DataFrame) -> pd.DataFrame:
    print_section("CLEANING: CRIME DATA")
 
    df = df.copy()
    original_len = len(df)

    df['provinsi'] = df['provinsi'].apply(normalize_province_name)
    df['tahun']       = df['tahun'].astype(int)
    df['crime_total'] = pd.to_numeric(df['crime_total'], errors='coerce')
 
    outlier_mask = df['crime_total'] > OUTLIER_MAX
    df['is_outlier'] = outlier_mask
 
    outliers = df[outlier_mask][['provinsi', 'tahun', 'crime_total']]
    
    if not outliers.empty:
        print(f"\n  Outlier ditemukan (crime_total > {OUTLIER_MAX:,}):")
        for _, row in outliers.iterrows():
            print(f"    → {row['provinsi']} {int(row['tahun'])}: {int(row['crime_total']):,}")
        df.loc[outlier_mask, 'crime_total'] = np.nan
        print(f"  Nilai outlier diganti NaN dan di-flag kolom 'is_outlier'")
    else:
        print_validation("Tidak ada outlier ekstrem", True)
 
    before = len(df)
    df = df.drop_duplicates(subset=['provinsi', 'tahun'], keep='last')
    dropped = before - len(df)
    print_validation(
        "Duplikat", dropped == 0,
        f"{dropped} baris duplikat dihapus" if dropped else "tidak ada duplikat"
    )

    pivot = df.pivot_table(
        index='provinsi', columns='tahun', values='crime_total', aggfunc='first'
    )
    years = sorted(df['tahun'].unique())
    print(f"\n  Coverage per tahun (jumlah provinsi yang punya data):")
    for yr in years:
        count = pivot[yr].notna().sum() if yr in pivot.columns else 0
        flag  = "✅" if count >= 30 else "⚠️ "
        print(f"    {flag} {yr}: {count} / 34 provinsi")
 
    unknown = set(df['provinsi'].unique()) - PROVINSI_RESMI
    print_validation(
        "Semua nama provinsi valid",
        len(unknown) == 0,
        f"Tidak dikenal: {unknown}" if unknown else ""
    )

    df = df.sort_values(['provinsi', 'tahun']).reset_index(drop=True)
 
    print(f"\n  Shape sebelum : ({original_len}, {3})")
    print(f"  Shape sesudah : {df.shape}")
    return df

## Cleaning Populasi

In [6]:
def clean_population(df: pd.DataFrame) -> pd.DataFrame:
    print_section("CLEANING: POPULASI")
 
    df = df.copy()
 
    df['provinsi'] = df['provinsi'].apply(normalize_province_name)
 
    df['tahun']      = df['tahun'].astype(int)
    df['population'] = pd.to_numeric(df['population'], errors='coerce')
    if 'population_density' in df.columns:
        df['population_density'] = pd.to_numeric(df['population_density'], errors='coerce')

    df = df.drop_duplicates(subset=['provinsi', 'tahun'], keep='last')

    df_2020 = df[df['tahun'] == 2020][['provinsi', 'population', 'population_density']].copy()
 
    extrapolated = []
    for yr in range(2015, 2020):
        n = 2020 - yr   
        df_yr = df_2020.copy()
        df_yr['population'] = (df_yr['population'] / ((1 + POPULATION_GROWTH_RATE) ** n)).round(0).astype(int)
        if 'population_density' in df_yr.columns:
            df_yr['population_density'] = (df_yr['population_density'] / ((1 + POPULATION_GROWTH_RATE) ** n)).round(1)
        df_yr['tahun']         = yr
        df_yr['is_extrapolated'] = True
        extrapolated.append(df_yr)
 
    df['is_extrapolated'] = False
    df_all = pd.concat([df] + extrapolated, ignore_index=True)
    df_all = df_all[df_all['tahun'].between(2015, 2026)]
    df_all = df_all.sort_values(['provinsi', 'tahun']).reset_index(drop=True)
 
    extrapolated_count = df_all['is_extrapolated'].sum()
    print_validation(
        f"Ekstrapolasi 2015–2019",
        extrapolated_count > 0,
        f"{extrapolated_count} rows dari growth rate {POPULATION_GROWTH_RATE*100:.2f}%/tahun"
    )
 
    weird = df_all[(df_all['population'] < 100_000) | (df_all['population'] > 60_000_000)]
    print_validation(
        "Nilai populasi dalam batas wajar",
        len(weird) == 0,
        f"{len(weird)} nilai anomali" if not weird.empty else ""
    )
 
    print(f"\n  Tahun tersedia : {sorted(df_all['tahun'].unique())}")
    print(f"  Provinsi       : {df_all['provinsi'].nunique()}")
    print(f"  Total rows     : {len(df_all)}")
    return df_all

## Cleaning Pengangguran

In [7]:
def clean_unemployment(df: pd.DataFrame) -> pd.DataFrame:
    print_section("CLEANING: PENGANGGURAN (TPT)")
 
    df = df.copy()
 
    df['provinsi'] = df['provinsi'].apply(normalize_province_name)
 
    df['tahun'] = df['tahun'].astype(int)
    for col in ['unemployment_rate_feb', 'unemployment_rate_aug', 'unemployment_rate_avg']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
 
    df = df.drop_duplicates(subset=['provinsi', 'tahun'], keep='last')

    if 'unemployment_rate_avg' in df.columns:
        weird = df[
            df['unemployment_rate_avg'].notna() &
            ~df['unemployment_rate_avg'].between(0.5, 20)
        ]
        print_validation(
            "Nilai TPT dalam batas wajar (0.5%–20%)",
            len(weird) == 0,
            f"{len(weird)} nilai anomali" if not weird.empty else ""
        )
 
    print(f"\n  Catatan: Data TPT hanya tersedia tahun {sorted(df['tahun'].unique())}")
    print(f"  Untuk 2015–2023, kolom unemployment_rate akan NULL di merged dataset")
    print(f"  Provinsi : {df['provinsi'].nunique()}")
    print(f"  Total rows: {len(df)}")
 
    df = df.sort_values(['provinsi', 'tahun']).reset_index(drop=True)
    return df

## Merge & Hitung Crime Rate

In [8]:
def merge_all(
    df_crime: pd.DataFrame,
    df_pop:   pd.DataFrame,
    df_unemp: pd.DataFrame,
) -> pd.DataFrame:
    print_section("MERGE SEMUA DATASET")

    df = pd.merge(
        df_crime,
        df_pop[['provinsi', 'tahun', 'population', 'population_density',
                'is_extrapolated']].rename(columns={'is_extrapolated': 'pop_extrapolated'}),
        on=['provinsi', 'tahun'],
        how='left'
    )

    df['crime_rate_per100k'] = (
        df['crime_total'] / df['population'] * 100_000
    ).round(1)

    df = pd.merge(
        df,
        df_unemp[['provinsi', 'tahun', 'unemployment_rate_avg']].rename(
            columns={'unemployment_rate_avg': 'unemployment_rate'}
        ),
        on=['provinsi', 'tahun'],
        how='left'
    )

    cols = [
        'provinsi', 'tahun',
        'crime_total', 'is_outlier',
        'population', 'population_density', 'pop_extrapolated',
        'crime_rate_per100k',
        'unemployment_rate',
    ]
    df = df[[c for c in cols if c in df.columns]]
    df = df.sort_values(['provinsi', 'tahun']).reset_index(drop=True)

    total = len(df)
    has_rate   = df['crime_rate_per100k'].notna().sum()
    has_unemp  = df['unemployment_rate'].notna().sum()
    has_pop    = df['population'].notna().sum()
 
    print(f"\n  Total rows         : {total}")
    print(f"  Provinsi           : {df['provinsi'].nunique()}")
    print(f"  Tahun              : {df['tahun'].min()}–{df['tahun'].max()}")
    print(f"  crime_rate_per100k : {has_rate}/{total} terisi ({has_rate/total*100:.1f}%)")
    print(f"  population         : {has_pop}/{total} terisi ({has_pop/total*100:.1f}%)")
    print(f"  unemployment_rate  : {has_unemp}/{total} terisi ({has_unemp/total*100:.1f}%)")
 
    print(f"\n  Top 5 crime rate tertinggi (2024):")
    sample = (
        df[df['tahun'] == 2024]
        .dropna(subset=['crime_rate_per100k'])
        .sort_values('crime_rate_per100k', ascending=False)
        .head(5)[['provinsi', 'crime_total', 'population', 'crime_rate_per100k', 'unemployment_rate']]
    )
    print(sample.to_string(index=False))
 
    return df
 

## Main

In [9]:
def main():
    print("\n" + "=" * 60)
    print("TAHAP 2 — DATA CLEANING & VALIDATION")
    print(f"Input  : {EXTRACTED_DIR}")
    print(f"Output : {CLEANED_DIR}")
    print("=" * 60)

    df_crime = pd.read_csv(EXTRACTED_DIR / "crime_raw.csv")
    df_pop   = pd.read_csv(EXTRACTED_DIR / "population_raw.csv")
    df_unemp = pd.read_csv(EXTRACTED_DIR / "unemployment_raw.csv")
 
    print(f"\nLoaded:")
    print(f"  crime_raw.csv        : {len(df_crime)} rows")
    print(f"  population_raw.csv   : {len(df_pop)} rows")
    print(f"  unemployment_raw.csv : {len(df_unemp)} rows")

    df_crime_clean = clean_crime(df_crime)
    df_pop_clean   = clean_population(df_pop)
    df_unemp_clean = clean_unemployment(df_unemp)

    df_merged = merge_all(df_crime_clean, df_pop_clean, df_unemp_clean)
 
    df_crime_clean.to_csv(CLEANED_DIR / "crime_cleaned.csv",        index=False)
    df_pop_clean.to_csv(  CLEANED_DIR / "population_cleaned.csv",   index=False)
    df_unemp_clean.to_csv(CLEANED_DIR / "unemployment_cleaned.csv", index=False)
    df_merged.to_csv(     CLEANED_DIR / "crime_merged.csv",         index=False)
 
    print_section("SUMMARY OUTPUT")
    print(f"  ✅ crime_cleaned.csv        → {len(df_crime_clean)} rows")
    print(f"  ✅ population_cleaned.csv   → {len(df_pop_clean)} rows")
    print(f"  ✅ unemployment_cleaned.csv → {len(df_unemp_clean)} rows")
    print(f"  ✅ crime_merged.csv         → {len(df_merged)} rows  ← dataset utama")
    print(f"\nTahap 2 selesai. Lanjut ke: src/03_load_to_sql.py")
 
 
if __name__ == "__main__":
    main()


TAHAP 2 — DATA CLEANING & VALIDATION
Input  : /Users/paulinadevinawijaya/Desktop/CRIME IN INDONESIA PORT/data/extracted
Output : /Users/paulinadevinawijaya/Desktop/CRIME IN INDONESIA PORT/data/cleaned

Loaded:
  crime_raw.csv        : 331 rows
  population_raw.csv   : 227 rows
  unemployment_raw.csv : 114 rows

CLEANING: CRIME DATA

  Outlier ditemukan (crime_total > 200,000):
    → Bengkulu 2016: 203,687
  Nilai outlier diganti NaN dan di-flag kolom 'is_outlier'
  ✅ Duplikat — tidak ada duplikat

  Coverage per tahun (jumlah provinsi yang punya data):
    ✅ 2015: 32 / 34 provinsi
    ✅ 2016: 31 / 34 provinsi
    ✅ 2017: 32 / 34 provinsi
    ✅ 2018: 32 / 34 provinsi
    ✅ 2019: 33 / 34 provinsi
    ✅ 2020: 34 / 34 provinsi
    ✅ 2021: 34 / 34 provinsi
    ✅ 2022: 34 / 34 provinsi
    ✅ 2023: 34 / 34 provinsi
    ✅ 2024: 34 / 34 provinsi
  ✅ Semua nama provinsi valid

  Shape sebelum : (331, 3)
  Shape sesudah : (331, 4)

CLEANING: POPULASI
  ✅ Ekstrapolasi 2015–2019 — 155 rows dari gr